Confusion Matrix (VGG16 - Before):
[[34837  2162]
 [ 1962  5044]]

Metrics Check:
Accuracy: 0.9062833768889899
Precision: 0.6999722453510963
Recall: 0.7199543248644019
F1: 0.7098226850548832


In [3]:
"""
============================================================
TIER 1 STATISTICAL ANALYSIS
Evaluation of Explainable AI (XAI) in Medical Imaging
============================================================

This script performs all 6 Tier 1 analyses:
  1. Descriptive statistics  (mean, median, SD, min, max, IQR)
  2. Frequency counts & %    (map preference per case)
  3. Friedman test           (compare Grad-CAM vs LIME vs SHAP)
  4. Wilcoxon signed-rank    (post-hoc pairwise + Bonferroni)
  5. Cronbach's alpha        (trust subscale Q20–Q23)
  6. Effect sizes            (eta-squared + r per test)

Requirements:
    pip install pandas openpyxl scipy numpy pingouin

Usage:
    python tier1_statistical_analysis.py
"""

# ─────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import friedmanchisquare, wilcoxon
import warnings
warnings.filterwarnings("ignore")

# Optional: pingouin for Cronbach's alpha (cleaner API)
try:
    import pingouin as pg
    HAS_PINGOUIN = True
except ImportError:
    HAS_PINGOUIN = False
    print("[INFO] pingouin not installed — using manual Cronbach's alpha formula.\n"
          "       Install with: pip install pingouin\n")


# ─────────────────────────────────────────────
# CONFIGURATION — file path
# ─────────────────────────────────────────────
FILE_PATH = "C:\\Users\\APURBA ROY\\Downloads\\ev.xlsx"

# Column index map (0-based, matches the survey)
COL = {
    # Preference (categorical): which map was chosen
    "pref_c1": 3,   # Image 1 map preference
    "pref_c2": 7,   # Image 2 map preference
    "pref_c3": 11,  # Image 3 map preference

    # Likert ratings: Grad-CAM
    "gradcam_c1": 4,
    "gradcam_c2": 8,
    "gradcam_c3": 12,

    # Likert ratings: LIME
    "lime_c1": 5,
    "lime_c2": 9,
    "lime_c3": 13,

    # Likert ratings: SHAP
    "shap_c1":  6,
    "shap_c2":  10,
    "shap_c3":  14,

    # Trust subscale (Q20–Q23)
    "trust_q20": 20,  # Correctly highlighted region
    "trust_q21": 21,  # Made AI decision easier to understand
    "trust_q22": 22,  # Increased trust in AI system
    "trust_q23": 23,  # Helpful in real diagnostic setting
}

DIVIDER = "\n" + "=" * 65 + "\n"


# ─────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────
def load_data(path: str) -> pd.DataFrame:
    df = pd.read_excel(path)
    print(f"Data loaded: {df.shape[0]} respondents, {df.shape[1]} columns")
    return df


# ─────────────────────────────────────────────
# HELPER: pretty print a table
# ─────────────────────────────────────────────
def print_table(title: str, df_table: pd.DataFrame):
    print(f"\n  {title}")
    print("  " + "-" * 55)
    print(df_table.to_string(index=True).replace("\n", "\n  "))
    print()


# ════════════════════════════════════════════════════════════
# ANALYSIS 1 — DESCRIPTIVE STATISTICS
# ════════════════════════════════════════════════════════════
def analysis_1_descriptive(df: pd.DataFrame):
    print(DIVIDER)
    print("ANALYSIS 1: DESCRIPTIVE STATISTICS")
    print("  Purpose : Summarise ratings for each XAI method")
    print("            across all 3 cases (Likert scale 1–5)")
    print("  Output  : Mean, Median, SD, Min, Max, IQR per method")
    print(DIVIDER)

    # Build a tidy dictionary of all rating columns
    rating_groups = {
        "Grad-CAM — Case 1": df.iloc[:, COL["gradcam_c1"]],
        "Grad-CAM — Case 2": df.iloc[:, COL["gradcam_c2"]],
        "Grad-CAM — Case 3": df.iloc[:, COL["gradcam_c3"]],
        "LIME     — Case 1": df.iloc[:, COL["lime_c1"]],
        "LIME     — Case 2": df.iloc[:, COL["lime_c2"]],
        "LIME     — Case 3": df.iloc[:, COL["lime_c3"]],
        "SHAP     — Case 1": df.iloc[:, COL["shap_c1"]],
        "SHAP     — Case 2": df.iloc[:, COL["shap_c2"]],
        "SHAP     — Case 3": df.iloc[:, COL["shap_c3"]],
    }

    rows = []
    for label, series in rating_groups.items():
        s = series.dropna()
        q1, q3 = np.percentile(s, [25, 75])
        rows.append({
            "Variable":  label,
            "N":         int(s.count()),
            "Mean":      round(s.mean(), 3),
            "Median":    round(s.median(), 3),
            "SD":        round(s.std(ddof=1), 3),
            "Min":       int(s.min()),
            "Max":       int(s.max()),
            "IQR":       round(q3 - q1, 2),
        })

    result_df = pd.DataFrame(rows).set_index("Variable")
    print_table("Descriptive statistics — XAI rating scores (1–5 Likert)", result_df)

    # Also compute per-method averages across all 3 cases
    print("  ── Averaged across all 3 cases ──")
    agg_rows = []
    for method, cols in [
        ("Grad-CAM", ["gradcam_c1", "gradcam_c2", "gradcam_c3"]),
        ("LIME",     ["lime_c1",    "lime_c2",    "lime_c3"]),
        ("SHAP",     ["shap_c1",    "shap_c2",    "shap_c3"]),
    ]:
        combined = pd.concat([df.iloc[:, COL[c]] for c in cols]).dropna()
        q1, q3 = np.percentile(combined, [25, 75])
        agg_rows.append({
            "Method":  method,
            "N obs":   int(combined.count()),
            "Mean":    round(combined.mean(), 3),
            "Median":  round(combined.median(), 3),
            "SD":      round(combined.std(ddof=1), 3),
            "IQR":     round(q3 - q1, 2),
        })

    agg_df = pd.DataFrame(agg_rows).set_index("Method")
    print_table("Aggregated descriptives (all 3 cases combined)", agg_df)

    print("  Interpretation hint:")
    print("  Higher mean = respondents felt the method highlighted")
    print("  the diagnostic region better (scale 1=poor, 5=excellent).")


# ════════════════════════════════════════════════════════════
# ANALYSIS 2 — FREQUENCY COUNTS & PERCENTAGES
# ════════════════════════════════════════════════════════════
def analysis_2_frequency(df: pd.DataFrame):
    print(DIVIDER)
    print("ANALYSIS 2: FREQUENCY COUNTS & PERCENTAGES")
    print("  Purpose : Show which XAI map respondents preferred")
    print("            most often in each of the 3 clinical cases")
    print("  Columns : Q3, Q7, Q11 (categorical map selection)")
    print(DIVIDER)

    cases = {
        "Case 1 (Q3) — abnormality map": df.iloc[:, COL["pref_c1"]],
        "Case 2 (Q7) — clinical region":  df.iloc[:, COL["pref_c2"]],
        "Case 3 (Q11) — disease area":    df.iloc[:, COL["pref_c3"]],
    }

    all_rows = []
    for case_name, series in cases.items():
        n_total = series.dropna().shape[0]
        vc = series.value_counts()
        print(f"\n  {case_name}  (n={n_total})")
        print("  " + "-" * 45)
        for label, count in vc.items():
            pct = count / n_total * 100
            bar = "█" * int(pct / 5)   # simple bar, each █ = ~5%
            print(f"  {str(label):<15}  {count:>3}  ({pct:5.1f}%)  {bar}")
            all_rows.append({
                "Case":       case_name,
                "Map chosen": label,
                "Count":      count,
                "Percent":    round(pct, 1),
            })

    print("\n  Full table:")
    freq_df = pd.DataFrame(all_rows)
    print("  " + freq_df.to_string(index=False).replace("\n", "\n  "))

    print("\n  Interpretation hint:")
    print("  The map chosen most often reflects the clinically")
    print("  preferred explanation method for each case.")


# ════════════════════════════════════════════════════════════
# ANALYSIS 3 — FRIEDMAN TEST
# ════════════════════════════════════════════════════════════
def analysis_3_friedman(df: pd.DataFrame):
    print(DIVIDER)
    print("ANALYSIS 3: FRIEDMAN TEST")
    print("  Purpose : Non-parametric repeated-measures test to check")
    print("            whether Grad-CAM, LIME, and SHAP ratings differ")
    print("            significantly within the same respondents.")
    print("  Why not ANOVA? Likert data is ordinal, not interval.")
    print("                 Friedman is the correct non-parametric alt.")
    print(DIVIDER)

    cases = [
        ("Case 1", COL["gradcam_c1"], COL["lime_c1"], COL["shap_c1"]),
        ("Case 2", COL["gradcam_c2"], COL["lime_c2"], COL["shap_c2"]),
        ("Case 3", COL["gradcam_c3"], COL["lime_c3"], COL["shap_c3"]),
    ]

    friedman_results = []

    for case_name, g_col, l_col, s_col in cases:
        # Drop rows with any missing value across the 3 methods
        case_df = df.iloc[:, [g_col, l_col, s_col]].dropna()
        g = case_df.iloc[:, 0].values
        l = case_df.iloc[:, 1].values
        s = case_df.iloc[:, 2].values
        n = len(g)

        # Friedman test
        stat, p = friedmanchisquare(g, l, s)

        # Effect size: Kendall's W
        # W = chi2 / (n * (k-1))   where k = number of conditions
        k = 3
        W = stat / (n * (k - 1))

        # Interpretation of W
        if W < 0.1:
            w_interp = "negligible"
        elif W < 0.3:
            w_interp = "small"
        elif W < 0.5:
            w_interp = "moderate"
        else:
            w_interp = "large"

        sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))

        friedman_results.append({
            "Case":        case_name,
            "N":           n,
            "χ²":          round(stat, 4),
            "df":          k - 1,
            "p-value":     f"{p:.4f}",
            "Sig.":        sig,
            "Kendall W":   round(W, 4),
            "Strength":    w_interp,
        })

        print(f"\n  ── {case_name} ──")
        print(f"     n = {n} respondents")
        print(f"     Grad-CAM  mean={g.mean():.2f}  median={np.median(g):.1f}")
        print(f"     LIME      mean={l.mean():.2f}  median={np.median(l):.1f}")
        print(f"     SHAP      mean={s.mean():.2f}  median={np.median(s):.1f}")
        print(f"     Friedman  χ²({k-1}) = {stat:.4f},  p = {p:.4f}  {sig}")
        print(f"     Effect size: Kendall's W = {W:.4f}  ({w_interp})")

        if p < 0.05:
            print(f"     → Significant difference found — run post-hoc (Analysis 4).")
        else:
            print(f"     → No significant difference between methods for {case_name}.")

    print("\n  Summary table:")
    fr_df = pd.DataFrame(friedman_results).set_index("Case")
    print_table("Friedman test results", fr_df)

    print("  Significance codes: *** p<0.001 | ** p<0.01 | * p<0.05 | ns = not significant")
    print("  Kendall's W: 0–0.1 negligible | 0.1–0.3 small | 0.3–0.5 moderate | >0.5 large")


# ════════════════════════════════════════════════════════════
# ANALYSIS 4 — WILCOXON SIGNED-RANK + BONFERRONI
# ════════════════════════════════════════════════════════════
def analysis_4_wilcoxon(df: pd.DataFrame):
    print(DIVIDER)
    print("ANALYSIS 4: WILCOXON SIGNED-RANK POST-HOC TESTS")
    print("  Purpose : Pairwise comparison of XAI methods to find")
    print("            WHICH method is significantly better.")
    print("  Pairs   : Grad-CAM vs LIME | Grad-CAM vs SHAP | LIME vs SHAP")
    print("  Correction: Bonferroni (multiply p × 3 comparisons)")
    print("  Effect size: r = Z / sqrt(N)  where Z is from normal approx.")
    print(DIVIDER)

    cases = [
        ("Case 1", COL["gradcam_c1"], COL["lime_c1"], COL["shap_c1"]),
        ("Case 2", COL["gradcam_c2"], COL["lime_c2"], COL["shap_c2"]),
        ("Case 3", COL["gradcam_c3"], COL["lime_c3"], COL["shap_c3"]),
    ]
    pairs = [("Grad-CAM vs LIME", 0, 1),
             ("Grad-CAM vs SHAP", 0, 2),
             ("LIME vs SHAP",     1, 2)]

    n_comparisons = len(pairs)  # Bonferroni denominator

    all_results = []

    for case_name, g_col, l_col, s_col in cases:
        case_df = df.iloc[:, [g_col, l_col, s_col]].dropna()
        data = [case_df.iloc[:, 0].values,
                case_df.iloc[:, 1].values,
                case_df.iloc[:, 2].values]
        n = len(data[0])
        names = ["Grad-CAM", "LIME", "SHAP"]

        print(f"\n  ── {case_name} (n={n}) ──")

        for pair_name, i, j in pairs:
            x, y = data[i], data[j]

            # Wilcoxon signed-rank test
            try:
                stat, p_raw = wilcoxon(x, y, alternative="two-sided")
            except ValueError:
                # All differences are zero
                print(f"     {pair_name}: all differences = 0, test not applicable")
                continue

            # Bonferroni-corrected p-value (capped at 1.0)
            p_adj = min(p_raw * n_comparisons, 1.0)

            # Effect size r = Z / sqrt(N)
            # scipy wilcoxon returns the test statistic W.
            # We derive Z from the normal approximation:
            #   mu_W = n*(n+1)/4,  sigma_W = sqrt(n*(n+1)*(2n+1)/24)
            n_pairs = n
            mu_w  = n_pairs * (n_pairs + 1) / 4
            sig_w = np.sqrt(n_pairs * (n_pairs + 1) * (2 * n_pairs + 1) / 24)
            Z     = (stat - mu_w) / sig_w
            r     = abs(Z) / np.sqrt(n_pairs)

            # Interpret effect size r
            if r < 0.1:
                r_interp = "negligible"
            elif r < 0.3:
                r_interp = "small"
            elif r < 0.5:
                r_interp = "moderate"
            else:
                r_interp = "large"

            sig_raw = "***" if p_raw < 0.001 else ("**" if p_raw < 0.01 else ("*" if p_raw < 0.05 else "ns"))
            sig_adj = "***" if p_adj < 0.001 else ("**" if p_adj < 0.01 else ("*" if p_adj < 0.05 else "ns"))

            print(f"     {pair_name:<25}  W={stat:.1f}  p_raw={p_raw:.4f}{sig_raw}"
                  f"  p_adj={p_adj:.4f}{sig_adj}  r={r:.3f} ({r_interp})")

            # Which method is higher?
            diff = np.mean(x) - np.mean(y)
            direction = f"{names[i]} > {names[j]}" if diff > 0 else f"{names[j]} > {names[i]}"
            if p_adj < 0.05:
                print(f"       → Significant after correction: {direction}")

            all_results.append({
                "Case":       case_name,
                "Pair":       pair_name,
                "W":          round(stat, 2),
                "p (raw)":    round(p_raw, 4),
                "p (adj)":    round(p_adj, 4),
                "Sig. adj.":  sig_adj,
                "r":          round(r, 3),
                "Strength":   r_interp,
            })

    print("\n  Full results table:")
    res_df = pd.DataFrame(all_results).set_index(["Case", "Pair"])
    print_table("Wilcoxon post-hoc with Bonferroni correction", res_df)

    print("  Effect size r: <0.1 negligible | 0.1–0.3 small | 0.3–0.5 moderate | >0.5 large")
    print("  Bonferroni: raw p-value multiplied by 3 (number of pairs per case).")
    print("  Report p_adj in the paper — this is the corrected value.")


# ════════════════════════════════════════════════════════════
# ANALYSIS 5 — CRONBACH'S ALPHA (TRUST SUBSCALE Q20–Q23)
# ════════════════════════════════════════════════════════════
def analysis_5_cronbach(df: pd.DataFrame):
    print(DIVIDER)
    print("ANALYSIS 5: CRONBACH'S ALPHA — TRUST SUBSCALE")
    print("  Purpose : Verify that Q20–Q23 form a reliable scale.")
    print("            α ≥ 0.70 is the accepted threshold for journals.")
    print("  Items   :")
    print("    Q20 — The explanation correctly highlighted the region.")
    print("    Q21 — Made the AI decision easier to understand.")
    print("    Q22 — Increased my trust in the AI system.")
    print("    Q23 — Helpful in a real diagnostic setting.")
    print(DIVIDER)

    trust_cols = [COL["trust_q20"], COL["trust_q21"],
                  COL["trust_q22"], COL["trust_q23"]]
    trust_df = df.iloc[:, trust_cols].dropna()
    trust_df.columns = ["Q20", "Q21", "Q22", "Q23"]
    n = len(trust_df)
    k = trust_df.shape[1]

    print(f"  N complete cases : {n}")
    print(f"  Number of items  : {k}")

    # ── Method A: pingouin (preferred) ──────────────────────
    if HAS_PINGOUIN:
        alpha_result = pg.cronbach_alpha(data=trust_df)
        alpha = alpha_result[0]
        ci    = alpha_result[1]
        print(f"\n  Cronbach's α = {alpha:.4f}  (95% CI: {ci[0]:.4f} – {ci[1]:.4f})")
        print("  [Computed via pingouin]")

    # ── Method B: manual formula (always shown for transparency) ──
    # α = (k/(k-1)) * (1 - Σvar_items / var_total)
    item_variances = trust_df.var(ddof=1)
    total_variance = trust_df.sum(axis=1).var(ddof=1)
    alpha_manual   = (k / (k - 1)) * (1 - item_variances.sum() / total_variance)

    print(f"\n  Cronbach's α (manual formula) = {alpha_manual:.4f}")
    print(f"     k = {k}  items")
    print(f"     Sum of item variances = {item_variances.sum():.4f}")
    print(f"     Variance of total score = {total_variance:.4f}")
    print(f"     Formula: ({k}/({k}-1)) × (1 − {item_variances.sum():.4f} / {total_variance:.4f})")

    # Item-total correlations (important for reporting)
    print("\n  Item-total correlations (Spearman):")
    total_score = trust_df.sum(axis=1)
    for col in trust_df.columns:
        r, p = stats.spearmanr(trust_df[col], total_score)
        print(f"     {col}: r = {r:.4f},  p = {p:.4f}")

    # Item descriptives
    print("\n  Item-level descriptives:")
    item_desc = trust_df.describe().T[["mean", "std", "min", "max"]]
    item_desc.columns = ["Mean", "SD", "Min", "Max"]
    item_desc = item_desc.round(3)
    print_table("Trust subscale (Q20–Q23) item stats", item_desc)

    # Interpretation
    print("  Interpretation:")
    alpha_val = alpha_manual
    if alpha_val >= 0.90:
        level = "excellent (≥ 0.90)"
    elif alpha_val >= 0.80:
        level = "good (0.80–0.89)"
    elif alpha_val >= 0.70:
        level = "acceptable (0.70–0.79)"
    elif alpha_val >= 0.60:
        level = "questionable (0.60–0.69)"
    else:
        level = "poor (< 0.60) — do NOT aggregate into a composite score"

    print(f"  α = {alpha_val:.4f} → {level}")
    if alpha_val >= 0.70:
        print("  → Items can be aggregated into a composite TRUST SCORE.")
        trust_score = trust_df.mean(axis=1)
        print(f"     Composite trust score  mean = {trust_score.mean():.3f},  "
              f"SD = {trust_score.std():.3f}")


# ════════════════════════════════════════════════════════════
# ANALYSIS 6 — EFFECT SIZES SUMMARY
# ════════════════════════════════════════════════════════════
def analysis_6_effect_sizes(df: pd.DataFrame):
    print(DIVIDER)
    print("ANALYSIS 6: EFFECT SIZE SUMMARY TABLE")
    print("  Purpose : Consolidate all effect sizes in one table for")
    print("            the paper. Journals require this alongside p-values.")
    print("  Metrics :")
    print("    η² (eta-squared) — for Friedman test")
    print("    r               — for Wilcoxon tests (r = Z/√N)")
    print("    Kendall's W     — overall concordance (Friedman)")
    print(DIVIDER)

    cases_config = [
        ("Case 1", COL["gradcam_c1"], COL["lime_c1"], COL["shap_c1"]),
        ("Case 2", COL["gradcam_c2"], COL["lime_c2"], COL["shap_c2"]),
        ("Case 3", COL["gradcam_c3"], COL["lime_c3"], COL["shap_c3"]),
    ]

    rows = []

    for case_name, g_col, l_col, s_col in cases_config:
        case_df = df.iloc[:, [g_col, l_col, s_col]].dropna()
        g = case_df.iloc[:, 0].values
        l = case_df.iloc[:, 1].values
        s = case_df.iloc[:, 2].values
        n = len(g)
        k = 3

        # Friedman
        chi2, p_f = friedmanchisquare(g, l, s)

        # η² = χ² / (n × (k-1))   — commonly used for Friedman
        eta2 = chi2 / (n * (k - 1))

        # Kendall's W = χ² / (n × (k-1))  — same formula, different name
        # (they are equivalent for the Friedman case)
        W = eta2

        # Wilcoxon r for each pair
        pairs_data = [
            ("Grad-CAM vs LIME", g, l),
            ("Grad-CAM vs SHAP", g, s),
            ("LIME vs SHAP",     l, s),
        ]
        pair_rs = []
        for _, x, y in pairs_data:
            try:
                stat, _ = wilcoxon(x, y, alternative="two-sided")
                mu_w  = n * (n + 1) / 4
                sig_w = np.sqrt(n * (n + 1) * (2 * n + 1) / 24)
                Z     = (stat - mu_w) / sig_w
                r     = abs(Z) / np.sqrt(n)
            except ValueError:
                r = 0.0
            pair_rs.append(round(r, 3))

        rows.append({
            "Case":                  case_name,
            "Friedman χ²":           round(chi2, 3),
            "p (Friedman)":          f"{p_f:.4f}",
            "η² / Kendall W":        round(eta2, 3),
            "r: GC vs LIME":         pair_rs[0],
            "r: GC vs SHAP":         pair_rs[1],
            "r: LIME vs SHAP":       pair_rs[2],
        })

    es_df = pd.DataFrame(rows).set_index("Case")
    print_table("Effect size summary across all 3 cases", es_df)

    print("  Effect size benchmarks:")
    print("    η² (Friedman):   small=0.01 | medium=0.06 | large=0.14")
    print("    r  (Wilcoxon):   small=0.10 | medium=0.30 | large=0.50")
    print("    Kendall's W:     poor=0.0–0.2 | fair=0.2–0.4 | good>0.4")
    print()
    print("  How to report in your paper (example sentence):")
    print('  "A Friedman test revealed a statistically significant difference')
    print("   among XAI methods for Case 1 (χ²(2) = X.XX, p = .XXX, η² = .XX),")
    print('   with Grad-CAM rated significantly higher than SHAP')
    print('   (Wilcoxon Z, p_adj = .XXX, r = .XX, Bonferroni corrected)."')


# ════════════════════════════════════════════════════════════
# MAIN
# ════════════════════════════════════════════════════════════
def main():
    print("\n" + "=" * 65)
    print("  XAI MEDICAL IMAGING — TIER 1 STATISTICAL ANALYSIS")
    print("=" * 65)

    df = load_data(FILE_PATH)

    analysis_1_descriptive(df)
    analysis_2_frequency(df)
    analysis_3_friedman(df)
    analysis_4_wilcoxon(df)
    analysis_5_cronbach(df)
    analysis_6_effect_sizes(df)

    print(DIVIDER)
    print("ALL TIER 1 ANALYSES COMPLETE")
    print("  Save this output and paste tables into your results section.")
    print("  Recommended order in paper:")
    print("    1. Descriptive stats table")
    print("    2. Map preference frequency table")
    print("    3. Friedman test table (with η²)")
    print("    4. Wilcoxon post-hoc table (with Bonferroni p_adj + r)")
    print("    5. Cronbach's α in the Methods/Results section")
    print("    6. Effect size summary table in Appendix or Results")
    print(DIVIDER)


if __name__ == "__main__":
    main()


  XAI MEDICAL IMAGING — TIER 1 STATISTICAL ANALYSIS
Data loaded: 49 respondents, 25 columns


ANALYSIS 1: DESCRIPTIVE STATISTICS
  Purpose : Summarise ratings for each XAI method
            across all 3 cases (Likert scale 1–5)
  Output  : Mean, Median, SD, Min, Max, IQR per method



  Descriptive statistics — XAI rating scores (1–5 Likert)
  -------------------------------------------------------
                    N   Mean  Median     SD  Min  Max  IQR
  Variable                                                  
  Grad-CAM — Case 1  49  4.000     4.0  0.935    2    5  2.0
  Grad-CAM — Case 2  49  3.592     4.0  1.079    1    5  1.0
  Grad-CAM — Case 3  49  3.531     4.0  1.120    1    5  1.0
  LIME     — Case 1  49  3.408     3.0  0.864    1    5  1.0
  LIME     — Case 2  49  3.510     3.0  1.082    1    5  1.0
  LIME     — Case 3  49  3.408     3.0  1.019    1    5  1.0
  SHAP     — Case 1  49  3.041     3.0  1.207    1    5  2.0
  SHAP     — Case 2  49  2.918     3.0  1.205    